# SAT-14B · Blinded audit, step 2: agreement, per-mode rates, corrected labels, re-score  (CPU, ~20-40 min)

**Attach:** `sat-ext` + `sat-splits` + `satelite-coco` + **`sat-audit-key`** (14A) + **`sat-audit-verdicts`** (filled
`verdicts_A.csv`, optional `verdicts_B.csv`, optional `verdicts_C_LLM.csv` = a model-assisted extra reviewer, reported only) + every results Dataset (`sat-results-*`, the same list as notebook 13B).
**Accelerator:** None. Run all / Save Version. Do **not** attach the notebook-13 outputs (14 and 13 are alternatives).

After it finishes - produces for the paper: **T19** missing-label share per inference mode (with 95 % CI), **T20** annotator agreement (kappa),
**T21** AP / precision / recall / F1 of standard vs SAHI on the original, corrected and "corrected + unsure" labels,
**T22** boxes added, **T23** every saved run re-scored, figures `fig_label_audit_examples.png` and `fig_label_audit_ap.png`.
Output -> New Dataset **`sat-results-14`**; re-run **04, 09, 12** with it attached (they use the corrected labels).

How to read it: if SAHI's AP is still below standard inference on the corrected labels, the paper's conclusion holds and is
now backed by the audit. If the gap closes or reverses, change the claim (the abstract and Section IV-B) to what T21 shows.

**Input setup** - right panel -> *Add Input* -> search the name -> *Add*. Links (your own Datasets are private: open them from the account that owns them, or share them with this account first):

* `sat-ext` - code (upload the sat_ext folder of the zip): https://www.kaggle.com/datasets/rifat4018/sat-ext
* `satelite-coco` - Roboflow COCO export, 640x640 (or download: universe.roboflow.com/rifat4018/aerial-object-detection-u3f5u): https://www.kaggle.com/datasets/rifat4018/satelite-coco
* `sat-splits` - output of notebook 01: https://www.kaggle.com/datasets/rifat4018/sat-splits
* `sat-audit-key` - output of notebook 14A: the private key results/audit/ (audit_key.csv, audit_meta.json): https://www.kaggle.com/datasets/rifat4018/sat-audit-key
* `sat-audit-verdicts` - the filled-in verdicts_A.csv (and verdicts_B.csv) from your annotators: https://www.kaggle.com/datasets/rifat4018/sat-audit-verdicts
* `sat-results-a`, `sat-results-b`, `sat-results-c`, `sat-results-d`, `sat-results-e`, `sat-results-f`, `sat-results-g`, `sat-results-07a`, `sat-results-07b`, `sat-results-07c`, `sat-results-07d`, `sat-results-07e`, `sat-results-07f`, `sat-results-07g`, `sat-results-07h`, `sat-results-07i`, `sat-results-07j`, `sat-results-07k`, `sat-results-07l`, `sat-results-08a`, `sat-results-08a1`, `sat-results-08b`, `sat-results-08b1`, `sat-results-08c`, `sat-results-08d`, `the output of notebook 03 (legacy weights + cross-dataset)` - outputs of earlier notebooks (Output tab of that notebook -> *New Dataset*, same name); search them under *Your Datasets*.
* Check the printed input tree in the next cell before running the rest.

In [ ]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

In [ ]:
utils.warn_if_gpu_wasted()
import cv2, shutil
import blindaudit as ba, dataset as ds, analysis, report, relabel

kp = utils.find_files("audit_key.csv")
assert kp, "attach the notebook 14A output as Dataset 'sat-audit-key' (results/audit/audit_key.csv)"
cands = pd.read_csv(kp[0])
meta = utils.load_json(os.path.join(os.path.dirname(kp[0]), "audit_meta.json"))
MODES, RID = meta["modes"], meta["run_id"]
pa_, pb_, pc_ = utils.find_files("verdicts_A.csv"), utils.find_files("verdicts_B.csv"), utils.find_files("verdicts_C_LLM.csv")
assert pa_, "attach the filled verdicts_A.csv as Dataset 'sat-audit-verdicts'"
va = ba.read_verdicts(pa_[0], cands["id"])
vb = ba.read_verdicts(pb_[0], cands["id"]) if pb_ else None
vc = ba.read_verdicts(pc_[0], cands["id"]) if pc_ else None     # optional extra, model-assisted reviewer (reported only)
print("integrity:", ba.integrity_check(va, vb))                  # stops if B was copied from A (or A from B)
final, ag = ba.consensus(cands, va, vb)
print("verdicts:", final.value_counts().to_dict(), "| annotator B reviewed:", 0 if vb is None else len(vb))
os.makedirs(report.PAPER_DIR, exist_ok=True)
report.save(ba.mode_table(cands, final, MODES), "T19_audit_missing_share_by_mode",
            f"Blinded audit of the apparent false positives of {RID} (conf >= 0.25): what the reviewers found, per inference mode. "
            "'only' rows: boxes produced by that mode alone. M = real but unlabelled object.")
report.save(pd.DataFrame([dict(boxes_checked_by_both=ag["n_both"], percent_agreement=round(100 * ag["agree"], 1) if ag["n_both"] else None,
                               kappa_4_codes=round(ag["kappa"], 2) if ag["n_both"] else None,
                               kappa_missing_vs_rest=round(ag["kappa_missing_vs_rest"], 2) if ag["n_both"] else None)]),
            "T20_audit_agreement", "Agreement of the two independent annotators on the doubly reviewed boxes (disagreements count as unsure).")
if vc is not None:
    report.save(ba.model_agreement(vc, va, vb), "T20b_audit_model_reviewer",
                "Agreement of an additional model-assisted reviewer (verdicts_C_LLM.csv) with the human annotators. Reported only; it never changes the labels.")

In [ ]:
runs = {}
for p in utils.find_files("eval_test.json"):
    f = os.path.dirname(p)
    if all(utils.dets_path(f, m) for m in MODES):
        runs[utils.load_json(p)["run_id"]] = f
assert RID in runs, f"run {RID} with saved detections is not attached"
dets = {m: analysis._dets(runs[RID], m) for m in MODES}
gt = analysis._gt_for("grouped")                      # ORIGINAL labels (13 / corrected splits must not be attached)
assert "corrected" not in str(gt.get("info", "")), "corrected labels are attached - remove the notebook-13 outputs (sat-results-13, sat-test-relabeled)"
gts = {"original": gt,
       "corrected": ba.corrected_gt(gt, cands, final),
       "corrected + unsure": ba.corrected_gt(gt, cands, final, include=("M", "U"))}
t21 = ba.score_versions(gts, dets, MODES)
report.save(t21.round(3), "T21_audit_rescore", f"{RID}: standard inference vs SAHI on the original labels, on the labels corrected with "
            "the audited missing objects, and with the unsure boxes also added (upper case). Same images, same detections.")
report.save(relabel.change_report(gt, gts["corrected"]), "T22_audit_boxes_added", "Ground-truth boxes per class and size before and after adding the audited missing objects.")
info = ds.materialize("grouped")
f1 = ba.fig_examples(cands, final, info["images"]["test"], os.path.join(report.PAPER_DIR, "fig_label_audit_examples.png"))
f2 = ba.fig_ap(t21[t21["labels"].isin(["original", "corrected"])], os.path.join(report.PAPER_DIR, "fig_label_audit_ap.png"))
print("figures:", f1, f2)
shutil.rmtree(info["root"], ignore_errors=True)

In [ ]:
gt_new = gts["corrected"]
relabel.write_corrected_splits(gt_new)
t = relabel.rescore_all(gt_new)
t.to_csv(os.path.join(report.PAPER_DIR, "T23_rescore_old_vs_new_audit.csv"), index=False)
s = t[t["mode"].isin(["standard", "sahi480", "sahi320", "sahi160", "tta", "res1024"])]
print(s.groupby(["model", "mode"])[["AP_old", "AP_new", "P_old", "P_new", "R_old", "R_new", "FP_old", "FP_new"]].mean().round(3).to_string())
z = shutil.make_archive(os.path.join(cfg.WORK_DIR, "results_14"), "zip", cfg.WORK_DIR, "results")
print(z)